# Apex Legends Steam Review Sentiment Analysis

## Project Overview
This notebook analyzes player sentiment in Apex Legends Steam reviews.
The objective is to classify reviews into **positive** and **negative** sentiment classes,
then evaluate model performance using standard classification metrics.


## 1) Import Libraries

In [ ]:
import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## 2) Load Dataset
Dataset source: scraped from Steam reviews for Apex Legends (App ID `1172470`).
The scraping script is provided in `scrape_apex_reviews.py`.

In [ ]:
df = pd.read_csv("review_1172470.csv")
df.head()

In [ ]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
df.info()

## 3) Data Cleaning and Preparation

In [ ]:
working_df = df.copy()

# Keep required columns only
working_df = working_df[["text", "label"]].dropna(subset=["text", "label"])

# Normalize label to integer: positive=True->1, negative=False->0
working_df["label"] = working_df["label"].astype(str).str.lower().map({"true": 1, "false": 0})
working_df = working_df.dropna(subset=["label"])
working_df["label"] = working_df["label"].astype(int)

def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

working_df["clean_text"] = working_df["text"].apply(clean_text)
working_df = working_df[working_df["clean_text"].str.len() > 0].reset_index(drop=True)

working_df.head()

In [ ]:
label_map = {1: "Positive", 0: "Negative"}
working_df["sentiment"] = working_df["label"].map(label_map)
working_df["sentiment"].value_counts()

## 4) Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(6, 4))
ax = sns.countplot(data=working_df, x="sentiment", order=["Positive", "Negative"], palette="viridis")
ax.set_title("Sentiment Distribution")
ax.set_xlabel("Sentiment")
ax.set_ylabel("Number of Reviews")
plt.tight_layout()
plt.show()

In [ ]:
working_df["review_length"] = working_df["clean_text"].str.split().str.len()

plt.figure(figsize=(8, 4))
sns.histplot(data=working_df, x="review_length", bins=50, kde=True)
plt.title("Review Length Distribution (Word Count)")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

## 5) Train/Test Split

In [ ]:
X = working_df["clean_text"]
y = working_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Train size: {len(X_train):,}")
print(f"Test size: {len(X_test):,}")

## 6) Modeling (Word2Vec + Naive Bayes)

In [ ]:
# Tokenize text for Word2Vec
X_train_tokens = X_train.apply(lambda text: text.split())
X_test_tokens = X_test.apply(lambda text: text.split())

# Train Word2Vec embeddings on training split only
w2v_model = Word2Vec(
    sentences=X_train_tokens,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    seed=RANDOM_STATE,
)

def sentence_vector(tokens, model):
    vectors = [model.wv[token] for token in tokens if token in model.wv]
    if not vectors:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

X_train_vec = np.vstack(X_train_tokens.apply(lambda tokens: sentence_vector(tokens, w2v_model)).values)
X_test_vec = np.vstack(X_test_tokens.apply(lambda tokens: sentence_vector(tokens, w2v_model)).values)

# MultinomialNB requires non-negative features
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train_vec)
X_test_scaled = scaler.transform(X_test_vec)

model = MultinomialNB()
model.fit(X_train_scaled, y_train)

## 7) Evaluation

In [ ]:
y_pred = model.predict(X_test_scaled)
acc = accuracy_score(y_test, y_pred)

print(f"Accuracy: {acc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Negative", "Positive"]))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Negative", "Positive"],
    yticklabels=["Negative", "Positive"],
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

## 8) Sample Predictions

In [ ]:
sample_reviews = pd.DataFrame({"review": X_test.sample(5, random_state=RANDOM_STATE).values})
sample_tokens = sample_reviews["review"].apply(lambda text: text.split())
sample_vectors = np.vstack(sample_tokens.apply(lambda tokens: sentence_vector(tokens, w2v_model)).values)
sample_scaled = scaler.transform(sample_vectors)
sample_reviews["prediction"] = pd.Series(model.predict(sample_scaled)).map(label_map)
sample_reviews

## 9) Key Takeaways
- This project demonstrates an end-to-end NLP workflow for sentiment classification.
- The final model uses Word2Vec embeddings with Multinomial Naive Bayes.
- The dataset was obtained via Steam review scraping (see `scrape_apex_reviews.py`).
- This notebook can be extended with hyperparameter tuning and model comparison.